# Election cleaning

# Load the raw election data

In [1]:
import pandas as pd
import numpy as np
import warnings

In [2]:
elections = pd.read_csv('/Users/matthewcavanaugh/Desktop/GitHub/us-presidential-elections-by-county/sources/Election 2000 2020 raw.csv')
elections

,year,state,state_po,county_name,county_fips,office,candidate,party,candidatevotes,totalvotes,version,mode
0,2000,ALABAMA,AL,AUTAUGA,1001.0,US PRESIDENT,AL GORE,DEMOCRAT,4942,17208,20220315,TOTAL
1,2000,ALABAMA,AL,AUTAUGA,1001.0,US PRESIDENT,GEORGE W. BUSH,REPUBLICAN,11993,17208,20220315,TOTAL
2,2000,ALABAMA,AL,AUTAUGA,1001.0,US PRESIDENT,RALPH NADER,GREEN,160,17208,20220315,TOTAL
3,2000,ALABAMA,AL,AUTAUGA,1001.0,US PRESIDENT,OTHER,OTHER,113,17208,20220315,TOTAL
4,2000,ALABAMA,AL,BALDWIN,1003.0,US PRESIDENT,AL GORE,DEMOCRAT,13997,56480,20220315,TOTAL
...,...,...,...,...,...,...,...,...,...,...,...,...
72612,2020,WYOMING,WY,WASHAKIE,56043.0,US PRESIDENT,DONALD J TRUMP,REPUBLICAN,3245,4032,20220315,TOTAL
72613,2020,WYOMING,WY,WESTON,56045.0,US PRESIDENT,JOSEPH R BIDEN JR,DEMOCRAT,360,3560,20220315,TOTAL
72614,2020,WYOMING,WY,WESTON,56045.0,US PRESIDENT,JO JORGENSEN,LIBERTARIAN,46,3560,20220315,TOTAL
72615,2020,WYOMING,WY,WESTON,56045.0,US PRESIDENT,OTHER,OTHER,47,3560,20220315,TOTAL


# EDA

In [3]:
elections.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72617 entries, 0 to 72616
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   year            72617 non-null  int64  
 1   state           72617 non-null  object 
 2   state_po        72617 non-null  object 
 3   county_name     72617 non-null  object 
 4   county_fips     72565 non-null  float64
 5   office          72617 non-null  object 
 6   candidate       72617 non-null  object 
 7   party           72617 non-null  object 
 8   candidatevotes  72617 non-null  int64  
 9   totalvotes      72617 non-null  int64  
 10  version         72617 non-null  int64  
 11  mode            72617 non-null  object 
dtypes: float64(1), int64(4), object(7)
memory usage: 6.6+ MB


In [4]:
elections['mode'].value_counts()

mode
TOTAL                   60583
ELECTION DAY             3737
ABSENTEE                 1995
PROVISIONAL              1832
ABSENTEE BY MAIL         1038
ONE STOP                  500
ADVANCED VOTING           477
PROV                      477
EARLY                     453
EARLY VOTE                450
FAILSAFE                  230
FAILSAFE PROVISIONAL      230
IN-PERSON ABSENTEE        230
MAIL                      145
2ND ABSENTEE              120
EARLY VOTING              120
Name: count, dtype: int64

In [5]:
elections.isnull().sum()

year               0
state              0
state_po           0
county_name        0
county_fips       52
office             0
candidate          0
party              0
candidatevotes     0
totalvotes         0
version            0
mode               0
dtype: int64

The county_fips codes act as an important unique identifier for every county equivalent. They will be key in merging with other datasets. So it is suspicious that some rows don't have a FIPS Code. Let's see them.

In [6]:
null_fips = elections[elections['county_fips'].isnull()]
null_fips

,year,state,state_po,county_name,county_fips,office,candidate,party,candidatevotes,totalvotes,version,mode
12452,2000,CONNECTICUT,CT,STATEWIDE WRITEIN,NaN,US PRESIDENT,AL GORE,DEMOCRAT,0,0,20220315,TOTAL
12453,2000,MAINE,ME,MAINE UOCAVA,NaN,US PRESIDENT,AL GORE,DEMOCRAT,0,0,20220315,TOTAL
12454,2000,RHODE ISLAND,RI,FEDERAL PRECINCT,NaN,US PRESIDENT,AL GORE,DEMOCRAT,0,0,20220315,TOTAL
12455,2000,CONNECTICUT,CT,STATEWIDE WRITEIN,NaN,US PRESIDENT,GEORGE W. BUSH,REPUBLICAN,0,0,20220315,TOTAL
12456,2000,MAINE,ME,MAINE UOCAVA,NaN,US PRESIDENT,GEORGE W. BUSH,REPUBLICAN,0,0,20220315,TOTAL
12457,2000,RHODE ISLAND,RI,FEDERAL PRECINCT,NaN,US PRESIDENT,GEORGE W. BUSH,REPUBLICAN,0,0,20220315,TOTAL
12458,2000,CONNECTICUT,CT,STATEWIDE WRITEIN,NaN,US PRESIDENT,RALPH NADER,GREEN,0,0,20220315,TOTAL
12459,2000,MAINE,ME,MAINE UOCAVA,NaN,US PRESIDENT,RALPH NADER,GREEN,0,0,20220315,TOTAL
12460,2000,RHODE ISLAND,RI,FEDERAL PRECINCT,NaN,US PRESIDENT,RALPH NADER,GREEN,0,0,20220315,TOTAL
12461,2000,CONNECTICUT,CT,STATEWIDE WRITEIN,NaN,US PRESIDENT,OTHER,OTHER,0,0,20220315,TOTAL


# Unusual reporting statistics
From 2012 onward, three states (Rhode Island, Maine, and Connecticut) have reported special types of votes (overseas ballots, write-ins, etc.) separately without assigning them to any particular counties. In its current form, this data will not be usable for county-level analysis. Here is how I will deal with this problem:
1. Take the sums of these unusual votes by state and year.
2. Assign votes proportionally to each county, based on the percentage of statewide votes it cast
3. Assign votes proportionally to candidates within the county, based on each candidate's vote share.
4. Add allocated votes to the original sums by party/county/year.
5. Drop the affected states from the original dataset, and merge the cleaned data.

We will temporarily drop these special values from the main dataset after splitting them off. We will rejoin once we have calculated the appropriate vote totals for candidates by county.

In [7]:
unusual_totals = null_fips.groupby(
    ['year', 'state', 'party', 'candidate']
)['candidatevotes'].sum().reset_index()

#nusual_totals = unusual_totals[~unusual_totals['year'].isin([2000, 2004, 2008])]
unusual_totals.rename(columns={'candidatevotes': 'candidate non county votes'}, inplace=True)
unusual_totals.head(5)

,year,state,party,candidate,candidate non county votes
0,2000,CONNECTICUT,DEMOCRAT,AL GORE,0
1,2000,CONNECTICUT,GREEN,RALPH NADER,0
2,2000,CONNECTICUT,OTHER,OTHER,0
3,2000,CONNECTICUT,REPUBLICAN,GEORGE W. BUSH,0
4,2000,MAINE,DEMOCRAT,AL GORE,0


For each of the three affected states, a single cell will successfully allocate non-county votes according to the process we've outlined. I've suppressed warnings to save space.

In [8]:
# Run this cell to allocate votes for Connecticut.

with warnings.catch_warnings():
    warnings.simplefilter("ignore")

    # Calculate each candidate's percentage of non-county votes by year
    non_county_connecticut = unusual_totals[unusual_totals['state'].str.contains('CONNECTICUT')]
    non_county_connecticut = non_county_connecticut.groupby(
        ['year', 'state', 'party', 'candidate']
    )['candidate non county votes'].sum().reset_index()
    non_county_connecticut['statewide non county votes'] = non_county_connecticut.groupby('year')['candidate non county votes'].transform('sum')
    non_county_connecticut['percent of non county votes'] = (
        non_county_connecticut['candidate non county votes'] / non_county_connecticut['statewide non county votes']
    ) * 100
    non_county_connecticut = non_county_connecticut.replace([np.inf, -np.inf], 0)
    non_county_connecticut['percent of non county votes'] = non_county_connecticut['percent of non county votes'].fillna(0)
    
    # Prepare county-level vote data
    counties_connecticut = elections[elections['state'].str.contains('CONNECTICUT')]
    counties_connecticut = counties_connecticut.drop_duplicates(subset=['county_name', 'year'], keep='first')
    counties_connecticut = counties_connecticut.dropna(subset=['county_fips'])
    counties_connecticut = counties_connecticut[['year', 'county_name', 'totalvotes']]
    
    # Calculate each county's percentage of statewide votes by year
    counties_connecticut['statewide votes'] = counties_connecticut.groupby('year')['totalvotes'].transform('sum')
    counties_connecticut['percent of statewide votes'] = (
        counties_connecticut['totalvotes'] / counties_connecticut['statewide votes']
    ) * 100
    
    # Merge total non-county votes by year into county-level data
    non_county_yearly_totals = non_county_connecticut.groupby('year')['candidate non county votes'].sum().rename('yearly_non_county_total').reset_index()
    counties_connecticut = counties_connecticut.merge(non_county_yearly_totals, on='year', how='left')
    
    # Allocate each county a precise number of non-county votes, based on its percentage of statewide votes cast by year
    counties_connecticut['proportional non county votes'] = (
        (counties_connecticut['percent of statewide votes'] / 100) * counties_connecticut['yearly_non_county_total']
    )
    counties_connecticut['proportional non county votes'] = counties_connecticut['proportional non county votes'].fillna(0).astype(int)
    counties_connecticut = counties_connecticut.fillna(0)
    
    # Calculate each candidate's percentage of the vote in each county by year
    connecticut_candidate_pcts = elections[elections['state'].str.contains('CONNECTICUT')]
    connecticut_candidate_pcts['statewide votes'] = connecticut_candidate_pcts.drop_duplicates(subset='county_name')['totalvotes'].sum()
    connecticut_candidate_pcts['pct of countywide vote'] = (
        connecticut_candidate_pcts['candidatevotes'] / connecticut_candidate_pcts['totalvotes']
    ) * 100
    
    # Merge estimated non-county votes into candidate data
    connecticut_candidate_pcts = connecticut_candidate_pcts.merge(
        counties_connecticut[['year', 'county_name', 'proportional non county votes']],
        on=['year', 'county_name'],
        how='left'
    )
    
    # Allocate non-county votes to candidates based on their county vote share
    connecticut_candidate_pcts['allocated non county votes'] = (
        (connecticut_candidate_pcts['pct of countywide vote'] / 100) *
        connecticut_candidate_pcts['proportional non county votes']
    ).fillna(0).round().astype(int)
    
    # Add allocated votes to original totals
    connecticut_candidate_pcts['candidatevotes'] += connecticut_candidate_pcts['allocated non county votes']
    
    # Finalize cleaned and adjusted dataset
    connecticut_adjusted = connecticut_candidate_pcts.copy()
    connecticut_adjusted.drop(columns=[
        'statewide votes',
        'pct of countywide vote',
        'proportional non county votes',
        'allocated non county votes'
    ], inplace=True)
    
    # Drop non-county records now that votes have been allocated to counties
    connecticut_adjusted = connecticut_adjusted.dropna(subset=['county_fips'])

print("Success! We have created a DataFrame called connecticut_adjusted that proportionally reallocates non-county votes in Connecticut.")

Success! We have created a DataFrame called connecticut_adjusted that proportionally reallocates non-county votes in Connecticut.


In [9]:
# Run this cell to allocate votes for Rhode Island.

with warnings.catch_warnings():
    warnings.simplefilter("ignore")

    # Calculate each candidate's percentage of non-county votes by year
    non_county_rhode_island = unusual_totals[unusual_totals['state'].str.contains('RHODE ISLAND')]
    non_county_rhode_island = non_county_rhode_island.groupby(
        ['year', 'state', 'party', 'candidate']
    )['candidate non county votes'].sum().reset_index()
    non_county_rhode_island['statewide non county votes'] = non_county_rhode_island.groupby('year')['candidate non county votes'].transform('sum')
    non_county_rhode_island['percent of non county votes'] = (
        non_county_rhode_island['candidate non county votes'] / non_county_rhode_island['statewide non county votes']
    ) * 100
    non_county_rhode_island = non_county_rhode_island.replace([np.inf, -np.inf], 0)
    non_county_rhode_island['percent of non county votes'] = non_county_rhode_island['percent of non county votes'].fillna(0)
    
    # Prepare county-level vote data
    counties_rhode_island = elections[elections['state'].str.contains('RHODE ISLAND')]
    counties_rhode_island = counties_rhode_island.drop_duplicates(subset=['county_name', 'year'], keep='first')
    counties_rhode_island = counties_rhode_island.dropna(subset=['county_fips'])
    counties_rhode_island = counties_rhode_island[['year', 'county_name', 'totalvotes']]
    
    # Calculate each county's percentage of statewide votes by year
    counties_rhode_island['statewide votes'] = counties_rhode_island.groupby('year')['totalvotes'].transform('sum')
    counties_rhode_island['percent of statewide votes'] = (
        counties_rhode_island['totalvotes'] / counties_rhode_island['statewide votes']
    ) * 100
    
    # Merge total non-county votes by year into county-level data
    non_county_yearly_totals = non_county_rhode_island.groupby('year')['candidate non county votes'].sum().rename('yearly_non_county_total').reset_index()
    counties_rhode_island = counties_rhode_island.merge(non_county_yearly_totals, on='year', how='left')
    
    # Allocate each county a precise number of non-county votes, based on its percentage of statewide votes cast by year
    counties_rhode_island['proportional non county votes'] = (
        (counties_rhode_island['percent of statewide votes'] / 100) * counties_rhode_island['yearly_non_county_total']
    )
    counties_rhode_island['proportional non county votes'] = counties_rhode_island['proportional non county votes'].fillna(0).astype(int)
    counties_rhode_island = counties_rhode_island.fillna(0)
    
    # Prepare candidate-level data
    rhode_island_candidate_pcts = elections[elections['state'].str.contains('RHODE ISLAND')]
    rhode_island_candidate_pcts['statewide votes'] = rhode_island_candidate_pcts.drop_duplicates(subset='county_name')['totalvotes'].sum()
    rhode_island_candidate_pcts['pct of countywide vote'] = (
        rhode_island_candidate_pcts['candidatevotes'] / rhode_island_candidate_pcts['totalvotes']
    ) * 100
    
    # Merge estimated non-county votes into candidate data
    rhode_island_candidate_pcts = rhode_island_candidate_pcts.merge(
        counties_rhode_island[['year', 'county_name', 'proportional non county votes']],
        on=['year', 'county_name'],
        how='left'
    )
    
    # Allocate non-county votes to candidates based on their county vote share
    rhode_island_candidate_pcts['allocated non county votes'] = (
        (rhode_island_candidate_pcts['pct of countywide vote'] / 100) *
        rhode_island_candidate_pcts['proportional non county votes']
    ).fillna(0).round().astype(int)
    
    # Add allocated votes to original totals
    rhode_island_candidate_pcts['candidatevotes'] += rhode_island_candidate_pcts['allocated non county votes']
    
    # Finalize cleaned and adjusted dataset
    rhode_island_adjusted = rhode_island_candidate_pcts.copy()
    rhode_island_adjusted.drop(columns=[
        'statewide votes',
        'pct of countywide vote',
        'proportional non county votes',
        'allocated non county votes'
    ], inplace=True)
    
    # Drop non-county records now that votes have been allocated to counties
    rhode_island_adjusted = rhode_island_adjusted.dropna(subset=['county_fips'])

print("Success! We have created a DataFrame called rhode_island_adjusted that proportionally reallocates non-county votes in Rhode Island.")

Success! We have created a DataFrame called rhode_island_adjusted that proportionally reallocates non-county votes in Rhode Island.


In [10]:
# Run this cell to allocate votes for Maine.

with warnings.catch_warnings():
    warnings.simplefilter("ignore")

    # Calculate each candidate's percentage of non-county votes by year
    non_county_maine = unusual_totals[unusual_totals['state'].str.contains('MAINE')]
    non_county_maine = non_county_maine.groupby(
        ['year', 'state', 'party', 'candidate']
    )['candidate non county votes'].sum().reset_index()
    non_county_maine['statewide non county votes'] = non_county_maine.groupby('year')['candidate non county votes'].transform('sum')
    non_county_maine['percent of non county votes'] = (
        non_county_maine['candidate non county votes'] / non_county_maine['statewide non county votes']
    ) * 100
    non_county_maine = non_county_maine.replace([np.inf, -np.inf], 0)
    non_county_maine['percent of non county votes'] = non_county_maine['percent of non county votes'].fillna(0)
    
    # Prepare county-level vote data
    counties_maine = elections[elections['state'].str.contains('MAINE')]
    counties_maine = counties_maine.drop_duplicates(subset=['county_name', 'year'], keep='first')
    counties_maine = counties_maine.dropna(subset=['county_fips'])
    counties_maine = counties_maine[['year', 'county_name', 'totalvotes']]
    
    # Calculate each county's percentage of statewide votes by year
    counties_maine['statewide votes'] = counties_maine.groupby('year')['totalvotes'].transform('sum')
    counties_maine['percent of statewide votes'] = (
        counties_maine['totalvotes'] / counties_maine['statewide votes']
    ) * 100
    
    # Merge total non-county votes by year into county-level data
    non_county_yearly_totals = non_county_maine.groupby('year')['candidate non county votes'].sum().rename('yearly_non_county_total').reset_index()
    counties_maine = counties_maine.merge(non_county_yearly_totals, on='year', how='left')
    
    # Allocate each county a precise number of non-county votes, based on its percentage of statewide votes cast by year
    counties_maine['proportional non county votes'] = (
        (counties_maine['percent of statewide votes'] / 100) * counties_maine['yearly_non_county_total']
    )
    counties_maine['proportional non county votes'] = counties_maine['proportional non county votes'].fillna(0).astype(int)
    counties_maine = counties_maine.fillna(0)
    
    # Prepare candidate-level data
    maine_candidate_pcts = elections[elections['state'].str.contains('MAINE')]
    maine_candidate_pcts['statewide votes'] = maine_candidate_pcts.drop_duplicates(subset='county_name')['totalvotes'].sum()
    maine_candidate_pcts['pct of countywide vote'] = (
        maine_candidate_pcts['candidatevotes'] / maine_candidate_pcts['totalvotes']
    ) * 100
    
    # Merge estimated non-county votes into candidate data
    maine_candidate_pcts = maine_candidate_pcts.merge(
        counties_maine[['year', 'county_name', 'proportional non county votes']],
        on=['year', 'county_name'],
        how='left'
    )
    
    # Allocate non-county votes to candidates based on their county vote share
    maine_candidate_pcts['allocated non county votes'] = (
        (maine_candidate_pcts['pct of countywide vote'] / 100) *
        maine_candidate_pcts['proportional non county votes']
    ).fillna(0).round().astype(int)
    
    # Add allocated votes to original totals
    maine_candidate_pcts['candidatevotes'] += maine_candidate_pcts['allocated non county votes']
    
    # Finalize cleaned and adjusted dataset
    maine_adjusted = maine_candidate_pcts.copy()
    maine_adjusted.drop(columns=[
        'statewide votes',
        'pct of countywide vote',
        'proportional non county votes',
        'allocated non county votes'
    ], inplace=True)
    
    # Drop non-county records now that votes have been allocated to counties
    maine_adjusted = maine_adjusted.dropna(subset=['county_fips'])

print("Success! We have created a DataFrame called maine_adjusted that proportionally reallocates non-county votes in Maine.")

Success! We have created a DataFrame called maine_adjusted that proportionally reallocates non-county votes in Maine.


We will drop the affected states from the original datasets, then merge it with the cleaned datasets for each state.

In [11]:
elections = elections[elections['state'] != 'CONNECTICUT']
elections = elections[elections['state'] != 'RHODE ISLAND']
elections = elections[elections['state'] != 'MAINE']

In [12]:
elections = pd.concat([elections, connecticut_adjusted, rhode_island_adjusted, maine_adjusted])

# Modes and candidate totals

In [13]:
elections['mode'].value_counts()

mode
TOTAL                   60531
ELECTION DAY             3737
ABSENTEE                 1995
PROVISIONAL              1832
ABSENTEE BY MAIL         1038
ONE STOP                  500
ADVANCED VOTING           477
PROV                      477
EARLY                     453
EARLY VOTE                450
FAILSAFE                  230
FAILSAFE PROVISIONAL      230
IN-PERSON ABSENTEE        230
MAIL                      145
2ND ABSENTEE              120
EARLY VOTING              120
Name: count, dtype: int64

Some counties split up candidate vote totals across multiple rows, breaking down categories such as Early Vote, Provision, and Election Day. The cell below will sum up candidate vote totals across all categories. 

In [14]:
# Sum up candidate votes across all types by county/year

# Group by relevant columns and sum candidatevotes
candidate_totals = elections.groupby(
    ['year', 'state', 'county_name', 'county_fips', 'office', 'party', 'candidate']
)['candidatevotes'].sum().reset_index()

# Optional: Sort values for readability
candidate_totals = candidate_totals.sort_values(by=['year', 'state', 'county_name', 'candidate'])

# Merge candidatevotes_total back into the original dataframe
elections = elections.merge(
    candidate_totals,
    on=['year', 'state', 'county_name', 'county_fips', 'office', 'party', 'candidate'],
    suffixes=('', '_total')
)


# Reformatting

Let's drop, rearrange, rename, and calculate new columns, and clean text.

In [15]:
# Clean and re-format the DataFrame

# Drop 'mode' as most of these have only a handful of value counts, and we have summed candidate totals 
elections.drop('mode', axis=1, inplace=True)

# Drop a few other redundant columns
elections.drop('office', axis=1, inplace=True)
elections.drop('version', axis=1, inplace=True)
elections.drop('state_po', axis=1, inplace=True)

# Rename columns
elections = elections.rename(columns={'candidatevotes_total': 'Votes'})
elections = elections.rename(columns={'year': 'Year'})
elections = elections.rename(columns={'state': 'State'})
elections = elections.rename(columns={'county_name': 'County'})
elections = elections.rename(columns={'state': 'State'})
elections = elections.rename(columns={'party': 'Party'})
elections = elections.rename(columns={'totalvotes': 'Total'})
elections = elections.rename(columns={'candidate': 'Candidate'})
elections = elections.rename(columns={'county_fips': 'FIPS Code'})

# Clean up text for aesthetic purposes
def clean_text(text):
    if isinstance(text, str):
        return text.title()  # Capitalizes the first letter of each word and makes the rest lowercase
    return text  # Return text unchanged if it's not a string
elections['County'] = elections['County'].apply(clean_text)
elections['Party'] = elections['Party'].apply(clean_text)
elections['State'] = elections['State'].apply(clean_text)
elections['Candidate'] = elections['Candidate'].apply(clean_text)

# Calculate each candidate's vote share percentage
elections['Percentage'] = round((elections['Votes'] / elections['Total'] * 100), 2)
elections

# Rearrange column order
elections = elections[['Year', 'State', 'County', 'FIPS Code', 'Party', 'Candidate', 'Votes', 'Percentage', 'Total']]
elections

,Year,State,County,FIPS Code,Party,Candidate,Votes,Percentage,Total
0,2000,Alabama,Autauga,1001.0,Democrat,Al Gore,4942,28.72,17208
1,2000,Alabama,Autauga,1001.0,Republican,George W. Bush,11993,69.69,17208
2,2000,Alabama,Autauga,1001.0,Green,Ralph Nader,160,0.93,17208
3,2000,Alabama,Autauga,1001.0,Other,Other,113,0.66,17208
4,2000,Alabama,Baldwin,1003.0,Democrat,Al Gore,13997,24.78,56480
...,...,...,...,...,...,...,...,...,...
72560,2020,Maine,York,23031.0,Democrat,Joseph R Biden Jr,71189,54.33,131042
72561,2020,Maine,York,23031.0,Green,Other,1259,0.96,131042
72562,2020,Maine,York,23031.0,Libertarian,Jo Jorgensen,2189,1.67,131042
72563,2020,Maine,York,23031.0,Other,Other,1588,1.21,131042


# Finding duplicates

In [16]:
def extract_duplicates(df):
    """
    Identifies and extracts exact duplicate rows from the dataframe.

    Parameters:
    df (pd.DataFrame): The original dataframe.

    Returns:
    pd.DataFrame: A dataframe containing only the duplicate rows.
    """
    duplicates = df[df.duplicated(keep=False)].copy()
    return duplicates

# Extract duplicates from your 'elections' DataFrame
duplicate_rows = extract_duplicates(elections)
duplicate_rows

,Year,State,County,FIPS Code,Party,Candidate,Votes,Percentage,Total
50418,2020,Arizona,Apache,4001.0,Democrat,Joseph R Biden Jr,23293,66.23,35172
50419,2020,Arizona,Apache,4001.0,Democrat,Joseph R Biden Jr,23293,66.23,35172
50420,2020,Arizona,Apache,4001.0,Democrat,Joseph R Biden Jr,23293,66.23,35172
50421,2020,Arizona,Apache,4001.0,Green,Other,0,0.00,35172
50422,2020,Arizona,Apache,4001.0,Green,Other,0,0.00,35172
...,...,...,...,...,...,...,...,...,...
71309,2020,Virginia,Winchester City,51840.0,Other,Other,69,0.57,12113
71310,2020,Virginia,Winchester City,51840.0,Other,Other,69,0.57,12113
71311,2020,Virginia,Winchester City,51840.0,Republican,Donald J Trump,5221,43.10,12113
71312,2020,Virginia,Winchester City,51840.0,Republican,Donald J Trump,5221,43.10,12113


Some duplicate rows are leftover from earlier data cleaning. This is nothing to worry about; these are the counties that split their voting modes - since we dropped that column, we can no longer see evidence of the differentiation.

In [17]:
elections = elections.drop_duplicates()

Pure duplicates should now be empty.

In [18]:
duplicate_rows = extract_duplicates(elections)
duplicate_rows

,Year,State,County,FIPS Code,Party,Candidate,Votes,Percentage,Total


But what about rows with conflicting vote totals among individual Counties?

Here's where we run into...
# The Virginia Problem


Virginia is a complicated case. It contains many Independent Cities which are treated as county equivalents.

A handful of Independent Cities share a name with a County despite being separate entities.
On the other hand, Independent Cities are not always listed consistently. In 2020, this dataset uses clarifying terms such as "City" or "County" to clear up confusion, but does not do so for 2000-2016 data.
This causes two types of problems:
1. The dataset doesn't know the difference between "Richmond County" and "Richmond City" despite both being separate entities.
2. The dataset believes that "Waynesboro" and "Waynesboro City" (among many examples) are different entities, despite actually being different renderings of the same entity.

In [19]:
# Find County Names that are represented by multiple rows with different vote totals.
def find_near_duplicates(df):
    # Group by identifying columns (excluding vote count and voting mode)
    group_cols = ['Year', 'State', 'County', 'Candidate', 'Party']
    
    # Count number of unique vote values per group
    grouped = df.groupby(group_cols)['Votes'].nunique().reset_index()
    
    # Keep only those with more than 1 unique vote count
    inconsistent = grouped[grouped['Votes'] > 1]
    
    # Merge back to original dataframe to get all rows involved
    near_duplicates = df.merge(inconsistent[group_cols], on=group_cols, how='inner')
    
    return near_duplicates
    
near_dupes = find_near_duplicates(elections)

print(f"Number of near-duplicate rows: {len(near_dupes)}")
print(f"Number of affected unique counties: {near_dupes[['State', 'County']].drop_duplicates().shape[0]}")
print(f"Number of affected unique states: {near_dupes[['State']].drop_duplicates().shape[0]}")

Number of near-duplicate rows: 160
Number of affected unique counties: 5
Number of affected unique states: 1


In [20]:
near_dupes['State'].value_counts()

State
Virginia    160
Name: count, dtype: int64

In [21]:
near_dupes['County'].value_counts()

County
Bedford     32
Fairfax     32
Franklin    32
Richmond    32
Roanoke     32
Name: count, dtype: int64

In [22]:
# Find County Names that are missing at least one election year
def find_incomplete_counties(df):
    required_years = {2000, 2004, 2008, 2012, 2016, 2020}
    
    # Group by state/county and collect unique years for each pair
    year_sets = df.groupby(['State', 'County'])['Year'].unique().reset_index()

    # Find those missing any required year
    year_sets['missing_years'] = year_sets['Year'].apply(lambda x: sorted(required_years - set(x)))
    incomplete = year_sets[year_sets['missing_years'].apply(len) > 0]

    return incomplete[['State', 'County', 'missing_years']]

incomplete_counties = find_incomplete_counties(elections) 

incomplete_counties['State'].value_counts()

State
Virginia        72
Maryland         2
South Dakota     2
Colorado         1
Name: count, dtype: int64

St. Mary's County, Maryland, is simply affected by spelling mistakes which can be quickly fixed. 

In [23]:
elections = elections.replace({"St. Mary'S": "St. Mary's"})
elections = elections.replace({"St Mary'S": "St. Mary's"})

South Dakota and Colorado present separate issues that will be addressed later. For now, we we will have to fix Virginia.

Thankfully, our FIPS Codes will allow us to distinguish these entities and standardize names.

The function below will clean up county names by setting all values in 'County' to whatever name is used in the most recent election, in our case 2020. For example, "Waynesboro" is listed as such in 2000-2016, and as "Waynesboro City" in 2020; after running this function, all values will now carry the latter name. Richmond City and Richmond County will also have their unique names for all years.

In [24]:
# Standardize County Names using FIPS Codes
def standardize_county_names(df):
    # Step 1: Identify FIPS codes with multiple county names
    multi_name_fips = (
        df.groupby('FIPS Code')['County']
        .nunique()
        .reset_index()
        .query('County > 1')['FIPS Code']
    )

    # Step 2: For each affected FIPS, find the most recent county name
    latest_names = (
        df[df['FIPS Code'].isin(multi_name_fips)]
        .sort_values('Year', ascending=False)
        .drop_duplicates(subset='FIPS Code')  # Keeps the most recent year only
        .set_index('FIPS Code')['County']
    )

    # Step 3: Map these corrected names back into the original dataframe
    df['County'] = df['FIPS Code'].map(latest_names).combine_first(df['County'])

    return df

# Apply the function to our original DataFrame
elections = standardize_county_names(elections)
elections

,Year,State,County,FIPS Code,Party,Candidate,Votes,Percentage,Total
0,2000,Alabama,Autauga,1001.0,Democrat,Al Gore,4942,28.72,17208
1,2000,Alabama,Autauga,1001.0,Republican,George W. Bush,11993,69.69,17208
2,2000,Alabama,Autauga,1001.0,Green,Ralph Nader,160,0.93,17208
3,2000,Alabama,Autauga,1001.0,Other,Other,113,0.66,17208
4,2000,Alabama,Baldwin,1003.0,Democrat,Al Gore,13997,24.78,56480
...,...,...,...,...,...,...,...,...,...
72560,2020,Maine,York,23031.0,Democrat,Joseph R Biden Jr,71189,54.33,131042
72561,2020,Maine,York,23031.0,Green,Other,1259,0.96,131042
72562,2020,Maine,York,23031.0,Libertarian,Jo Jorgensen,2189,1.67,131042
72563,2020,Maine,York,23031.0,Other,Other,1588,1.21,131042


Let's run another check to make sure our processes worked.

In [25]:
near_dupes = find_near_duplicates(elections)

print(f"Number of near-duplicate rows: {len(near_dupes)}")
print(f"Number of affected unique counties: {near_dupes[['State', 'County']].drop_duplicates().shape[0]}")
print(f"Number of affected unique states: {near_dupes[['State']].drop_duplicates().shape[0]}")

Number of near-duplicate rows: 32
Number of affected unique counties: 1
Number of affected unique states: 1


In [26]:
near_dupes['State'].value_counts()

State
Virginia    32
Name: count, dtype: int64

In [27]:
near_dupes['County'].value_counts()

County
Bedford    32
Name: count, dtype: int64

There is one more complication. An independent city named "Bedford" or "Bedford City" existed prior to 2013, when it was merged into Bedford County. Since it didn't exist in 2020, our function didn't catch it. Bedford City's FIPS code was 51515.0, so based on that we can drop it while keeping Bedford County.

In [28]:
elections = elections[elections['FIPS Code'] != 51515.0]

# The Alaska Problem
ALASKA - Alaska does not have counties in the traditional sense, but is instead broken into "organized" and "unorganized" boroughs. This would be fine except some data sources do not use these, but instead use "districts" with entirely different boundaries and names. Until or unless I can find a method to reconcile these differing statistical areas, I will have to exclude Alaska from the dataset.

In [29]:
elections = elections[elections['State'] != 'Alaska']

# Cleaning up county names

For full clarity, we will alter county names so that, for instance, "Autauga" is now rendered as "Autauga County", and capitalization is standardized. We have made exemptions for unusual cases such as Louisiana (which uses the term "Parish"), the District of Columbia, and others.

In [30]:
# Clean up county names

def format_county_names(df):
    def format_name(name, state):
        name_clean = name.strip().lower()
        
        # Explicit fixes for these Virginia counties
        if state == 'Virginia' and name_clean == 'james city':
            return 'James City County'
        elif state == 'Virginia' and name_clean == 'charles city':
            return 'Charles City County'
        
        # Leave as-is if already contains County, City, or District
        if 'City' in name or 'District' in name or 'County' in name:
            return name
        
        # Louisiana uses "Parish"
        elif state == 'Louisiana':
            return name + ' Parish'
        
        # Default: append "County"
        else:
            return name + ' County'
    
    df['County'] = df.apply(lambda row: format_name(row['County'], row['State']), axis=1)
    return df

format_county_names(elections)

,Year,State,County,FIPS Code,Party,Candidate,Votes,Percentage,Total
0,2000,Alabama,Autauga County,1001.0,Democrat,Al Gore,4942,28.72,17208
1,2000,Alabama,Autauga County,1001.0,Republican,George W. Bush,11993,69.69,17208
2,2000,Alabama,Autauga County,1001.0,Green,Ralph Nader,160,0.93,17208
3,2000,Alabama,Autauga County,1001.0,Other,Other,113,0.66,17208
4,2000,Alabama,Baldwin County,1003.0,Democrat,Al Gore,13997,24.78,56480
...,...,...,...,...,...,...,...,...,...
72560,2020,Maine,York County,23031.0,Democrat,Joseph R Biden Jr,71189,54.33,131042
72561,2020,Maine,York County,23031.0,Green,Other,1259,0.96,131042
72562,2020,Maine,York County,23031.0,Libertarian,Jo Jorgensen,2189,1.67,131042
72563,2020,Maine,York County,23031.0,Other,Other,1588,1.21,131042


# The Green/Libertarian Problem
The election dataset provides separate columns for the Green and Libertarian Parties, but they have not been on the ballot in every state in every election since 2000. On one hand, the absence of a minor party could cause the model to correctly assign higher vote shares to the major ones, but I also fear the model could make incorrect political inferences about why votes dropped to 0. For now, they have been rolled into a generic "Other" category for vote totals and percentages.

In [31]:
import pandas as pd

# Assuming your DataFrame is named 'elections' and has columns: 'Year', 'State', 'County', 'Party', 'Votes'

# Step 1: Standardize party names (optional but recommended)
elections['Party'] = elections['Party'].str.strip().str.title()

# Step 2: Map non-major parties to 'Other'
major_parties = ['Democrat', 'Republican']
elections['Party'] = elections['Party'].apply(lambda x: x if x in major_parties else 'Other')

# Step 3: Group by identifiers and sum votes
elections_grouped = elections.groupby(['Year', 'State', 'County', 'Party', 'FIPS Code'], as_index=False)['Votes'].sum()

In [32]:
elections_grouped

,Year,State,County,Party,FIPS Code,Votes
0,2000,Alabama,Autauga County,Democrat,1001.0,4942
1,2000,Alabama,Autauga County,Other,1001.0,273
2,2000,Alabama,Autauga County,Republican,1001.0,11993
3,2000,Alabama,Baldwin County,Democrat,1003.0,13997
4,2000,Alabama,Baldwin County,Other,1003.0,1611
...,...,...,...,...,...,...
56026,2020,Wyoming,Washakie County,Other,56043.0,136
56027,2020,Wyoming,Washakie County,Republican,56043.0,3245
56028,2020,Wyoming,Weston County,Democrat,56045.0,360
56029,2020,Wyoming,Weston County,Other,56045.0,93


# Pivot 

In [33]:
# Pivot to wide format and calculate new columns
election_pivot = elections_grouped.pivot(
    index=['Year', 'State', 'County', 'FIPS Code'],
    columns='Party',
    values='Votes'
).fillna(0).reset_index()

# Rename columns for clarity
election_pivot = election_pivot.rename(columns={
    'Democrat': 'Democrat Votes',
    'Republican': 'Republican Votes',
    'Other': 'Other Votes'
})

# Calculate total votes
election_pivot['Total Votes'] = (
    election_pivot['Democrat Votes'] +
    election_pivot['Republican Votes'] +
    election_pivot['Other Votes']
)

# Calculate percentages
election_pivot['Democrat Percentage'] = (election_pivot['Democrat Votes'] / election_pivot['Total Votes'] * 100).round(2)
election_pivot['Republican Percentage'] = (election_pivot['Republican Votes'] / election_pivot['Total Votes'] * 100).round(2)
election_pivot['Other Percentage'] = (election_pivot['Other Votes'] / election_pivot['Total Votes'] * 100).round(2)

# After pivoting and resetting index:
election_pivot.columns.name = None

# Reorder columns
election_pivot = election_pivot[['Year', 'State', 'County', 'FIPS Code', 'Democrat Votes', 'Democrat Percentage', 'Republican Votes', 'Republican Percentage', 'Other Votes', 'Other Percentage', 'Total Votes']]

# Calculate a few new columns
election_pivot['D-R margin'] = election_pivot['Democrat Percentage'] - election_pivot['Republican Percentage']
election_pivot['Two party vote share'] = election_pivot['Democrat Percentage'] + election_pivot['Republican Percentage']

election_pivot

,Year,State,County,FIPS Code,Democrat Votes,Democrat Percentage,Republican Votes,Republican Percentage,Other Votes,Other Percentage,Total Votes,D-R margin,Two party vote share
0,2000,Alabama,Autauga County,1001.0,4942,28.72,11993,69.69,273,1.59,17208,-40.97,98.41
1,2000,Alabama,Baldwin County,1003.0,13997,24.78,40872,72.37,1611,2.85,56480,-47.59,97.15
2,2000,Alabama,Barbour County,1005.0,5188,49.91,5096,49.02,111,1.07,10395,0.89,98.93
3,2000,Alabama,Bibb County,1007.0,2710,38.16,4273,60.17,118,1.66,7101,-22.01,98.33
4,2000,Alabama,Blount County,1009.0,4977,27.69,12667,70.48,329,1.83,17973,-42.79,98.17
...,...,...,...,...,...,...,...,...,...,...,...,...,...
18672,2020,Wyoming,Sweetwater County,56037.0,3823,22.89,12229,73.24,646,3.87,16698,-50.35,96.13
18673,2020,Wyoming,Teton County,56039.0,9848,66.60,4341,29.36,598,4.04,14787,37.24,95.96
18674,2020,Wyoming,Uinta County,56041.0,1591,16.82,7496,79.25,372,3.93,9459,-62.43,96.07
18675,2020,Wyoming,Washakie County,56043.0,651,16.15,3245,80.48,136,3.37,4032,-64.33,96.63


# Checking for nulls again

In [34]:
election_pivot.isnull().sum()

Year                     0
State                    0
County                   0
FIPS Code                0
Democrat Votes           0
Democrat Percentage      1
Republican Votes         0
Republican Percentage    1
Other Votes              0
Other Percentage         1
Total Votes              0
D-R margin               1
Two party vote share     1
dtype: int64

In [35]:
election_pivot[election_pivot.isnull().any(axis=1)]

,Year,State,County,FIPS Code,Democrat Votes,Democrat Percentage,Republican Votes,Republican Percentage,Other Votes,Other Percentage,Total Votes,D-R margin,Two party vote share
1503,2000,Missouri,Kansas City,2938000.0,0,NaN,0,NaN,0,NaN,0,NaN,NaN


# The Kansas City / Jackson County problem

For some reason, Kansas City, Missouri is listed separately despite not having county status. After 2000,  its numbers are split off from Jackson County, in which it is contained. Jackson County's post-2000, statistics, in turn, only reflect parts of the county outside of Kansas City. This is strange, as it appears to be the only instance of something like this being done in the dataset, but it is fixable.

In [36]:
election_pivot[election_pivot['County'].str.contains('Kansas City')]

,Year,State,County,FIPS Code,Democrat Votes,Democrat Percentage,Republican Votes,Republican Percentage,Other Votes,Other Percentage,Total Votes,D-R margin,Two party vote share
1503,2000,Missouri,Kansas City,2938000.0,0,NaN,0,NaN,0,NaN,0,NaN,NaN
4616,2004,Missouri,Kansas City,2938000.0,104625,73.98,36061,25.50,737,0.52,141423,48.48,99.48
7729,2008,Missouri,Kansas City,2938000.0,120102,78.39,31854,20.79,1263,0.82,153219,57.60,99.18
10842,2012,Missouri,Kansas City,2938000.0,105670,77.24,29509,21.57,1623,1.19,136802,55.67,98.81
13955,2016,Missouri,Kansas City,2938000.0,97735,76.00,24654,19.17,6212,4.83,128601,56.83,95.17
17068,2020,Missouri,Kansas City,2938000.0,107660,78.79,26393,19.32,2592,1.90,136645,59.47,98.11


In [37]:
jackson_mo = election_pivot[election_pivot['County'].str.contains('Jackson County')]
jackson_mo = jackson_mo[jackson_mo['State'].str.contains('Missouri')]
jackson_mo

,Year,State,County,FIPS Code,Democrat Votes,Democrat Percentage,Republican Votes,Republican Percentage,Other Votes,Other Percentage,Total Votes,D-R margin,Two party vote share
1499,2000,Missouri,Jackson County,29095.0,160419,58.96,104418,38.38,7225,2.66,272062,20.58,97.34
4612,2004,Missouri,Jackson County,29095.0,79029,45.27,94439,54.10,1102,0.63,174570,-8.83,99.37
7725,2008,Missouri,Jackson County,29095.0,90722,48.76,92833,49.90,2492,1.34,186047,-1.14,98.66
10838,2012,Missouri,Jackson County,29095.0,78283,44.79,93199,53.33,3282,1.88,174764,-8.54,98.12
13951,2016,Missouri,Jackson County,29095.0,71237,41.11,91557,52.84,10481,6.05,173275,-11.73,93.95
17064,2020,Missouri,Jackson County,29095.0,92182,46.93,100142,50.98,4094,2.08,196418,-4.05,97.91


The cell below will fix this data problem.

In [38]:
# Combine raw values for Kansas City and Jackson County, recalculate percentages, drop bad old data, merge new

# Save the FIPS Code for Jackson County, MO before any dropping
jackson_fips = election_pivot.loc[
    (election_pivot['State'] == 'Missouri') & 
    (election_pivot['County'] == 'Jackson County') & 
    (election_pivot['Year'] == 2000), 'FIPS Code'
].values[0]  # Safe since 2000 row exists and is correct

# Filter post-2000 Kansas City + Jackson County rows
mask_post2000 = (
    (election_pivot['State'] == 'Missouri') &
    (election_pivot['County'].isin(['Jackson County', 'Kansas City'])) &
    (election_pivot['Year'] > 2000)
)
kc_jackson_post2000 = election_pivot[mask_post2000].copy()

# Group by Year, summing vote totals
combined = kc_jackson_post2000.groupby('Year').agg({
    'Democrat Votes': 'sum',
    'Republican Votes': 'sum',
    'Other Votes': 'sum',
    'Total Votes': 'sum'
}).reset_index()

# Recalculate derived values
combined['Democrat Percentage'] = combined['Democrat Votes'] / combined['Total Votes'] * 100
combined['Republican Percentage'] = combined['Republican Votes'] / combined['Total Votes'] * 100
combined['Other Percentage'] = combined['Other Votes'] / combined['Total Votes'] * 100
combined['D-R margin'] = (combined['Democrat Votes'] - combined['Republican Votes']) / combined['Total Votes'] * 100
combined['Two party vote share'] = (
    (combined['Democrat Votes'] + combined['Republican Votes']) / combined['Total Votes'] * 100
)

# Add identifying columns
combined['State'] = 'Missouri'
combined['County'] = 'Jackson County'
combined['FIPS Code'] = jackson_fips

# Reorder columns to match original
combined = combined[election_pivot.columns]

# Drop original post-2000 Jackson County + Kansas City rows
election_pivot = election_pivot[~mask_post2000]

# Append combined values
election_pivot = pd.concat([election_pivot, combined], ignore_index=True)

# Optional: sort if needed
election_pivot = election_pivot.sort_values(by=['Year', 'State', 'County']).reset_index(drop=True)

# Drop empty Kansas City 2000 row
election_pivot = election_pivot[
    ~(
        (election_pivot['State'] == 'Missouri') &
        (election_pivot['County'] == 'Kansas City') &
        (election_pivot['Year'] == 2000)
    )
]

election_pivot

,Year,State,County,FIPS Code,Democrat Votes,Democrat Percentage,Republican Votes,Republican Percentage,Other Votes,Other Percentage,Total Votes,D-R margin,Two party vote share
0,2000,Alabama,Autauga County,1001.0,4942,28.72,11993,69.69,273,1.59,17208,-40.97,98.41
1,2000,Alabama,Baldwin County,1003.0,13997,24.78,40872,72.37,1611,2.85,56480,-47.59,97.15
2,2000,Alabama,Barbour County,1005.0,5188,49.91,5096,49.02,111,1.07,10395,0.89,98.93
3,2000,Alabama,Bibb County,1007.0,2710,38.16,4273,60.17,118,1.66,7101,-22.01,98.33
4,2000,Alabama,Blount County,1009.0,4977,27.69,12667,70.48,329,1.83,17973,-42.79,98.17
...,...,...,...,...,...,...,...,...,...,...,...,...,...
18667,2020,Wyoming,Sweetwater County,56037.0,3823,22.89,12229,73.24,646,3.87,16698,-50.35,96.13
18668,2020,Wyoming,Teton County,56039.0,9848,66.60,4341,29.36,598,4.04,14787,37.24,95.96
18669,2020,Wyoming,Uinta County,56041.0,1591,16.82,7496,79.25,372,3.93,9459,-62.43,96.07
18670,2020,Wyoming,Washakie County,56043.0,651,16.15,3245,80.48,136,3.37,4032,-64.33,96.63


In [39]:
jackson_mo = election_pivot[election_pivot['County'].str.contains('Jackson County')]
jackson_mo = jackson_mo[jackson_mo['State'].str.contains('Missouri')]
jackson_mo

,Year,State,County,FIPS Code,Democrat Votes,Democrat Percentage,Republican Votes,Republican Percentage,Other Votes,Other Percentage,Total Votes,D-R margin,Two party vote share
1499,2000,Missouri,Jackson County,29095.0,160419,58.960000,104418,38.380000,7225,2.660000,272062,20.580000,97.340000
4612,2004,Missouri,Jackson County,29095.0,183654,58.119642,130500,41.298383,1839,0.581975,315993,16.821259,99.418025
7724,2008,Missouri,Jackson County,29095.0,210824,62.141211,124687,36.751988,3755,1.106801,339266,25.389223,98.893199
10836,2012,Missouri,Jackson County,29095.0,183953,59.041423,122708,39.384272,4905,1.574305,311566,19.657151,98.425695
13948,2016,Missouri,Jackson County,29095.0,168972,55.973976,116211,38.496270,16693,5.529754,301876,17.477706,94.470246
17060,2020,Missouri,Jackson County,29095.0,199842,60.001261,126535,37.991311,6686,2.007428,333063,22.009950,97.992572


A red flag was Kansas City's FIPS Code being seven digits long. Any county equivalent's FIPS Code should be no more than five digits. Let's make sure there are no more weird cases like this.

In [40]:
election_pivot[election_pivot['FIPS Code'] >= 100000]

,Year,State,County,FIPS Code,Democrat Votes,Democrat Percentage,Republican Votes,Republican Percentage,Other Votes,Other Percentage,Total Votes,D-R margin,Two party vote share


In [41]:
election_pivot.isnull().sum()

Year                     0
State                    0
County                   0
FIPS Code                0
Democrat Votes           0
Democrat Percentage      0
Republican Votes         0
Republican Percentage    0
Other Votes              0
Other Percentage         0
Total Votes              0
D-R margin               0
Two party vote share     0
dtype: int64

# Missing county years

Are there any counties missing data for any years?

In [42]:
# Find missing county/year pairs
expected_years = {2000, 2004, 2008, 2012, 2016, 2020}

# Group by State and County
grouped = election_pivot.groupby(['State', 'County'])

# Identify counties with missing years
missing_data = []

for (state, county), group in grouped:
    present_years = set(group['Year'])
    missing_years = expected_years - present_years
    if missing_years:
        missing_data.append({
            'State': state,
            'County': county,
            'Missing Years': sorted(missing_years)
        })

# Convert to DataFrame
missing_df = pd.DataFrame(missing_data)

missing_df

,State,County,Missing Years
0,Colorado,Broomfield County,[2000]
1,South Dakota,Oglala Lakota County,"[2000, 2004, 2008, 2012]"
2,South Dakota,Shannon County,"[2016, 2020]"


Shannon County and Oglala Lakota County, South Dakota are the same; the former changed its name to the latter in 2015. Entries with the old name will be updated to the new for consistency. The county's FIPS Code also changed at this time; we will use the new one.

In [43]:
election_pivot.loc[
    (election_pivot['County'] == 'Shannon County') & 
    (election_pivot['State'] == 'South Dakota'),
    ['County', 'FIPS Code']
] = ['Oglala Lakota County', '46102']

/var/folders/cf/4zrkk0lx1nbdx9tpklpk4jc00000gn/T/ipykernel_6169/3454204712.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '46102' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  election_pivot.loc[


More pressing is...

# The Broomfield Problem

Broomfield County, Colorado did not exist in 2000. It was created in 2001 by breaking off pieces of four existing counties: Adams, Boulder, Jefferson, Weld. I see four possible ways to deal with this.
1. Simulate a prediction of 2000 for Broomfield County. But this would artifically inflate Colorado's statewide vote count as Broomfield's territory was previously spread across several pre-existing counties. 
2. Drop all real 2000 data for Colorado and reverse-interpolate all statistics from later years, including the populations of the counties Broomfield was created from. Or maybe this could be limited to just the affected counties.
3. Cut 2000 from the simulation altogether.
4. Drop Broomfield County. Since it has never cast enough votes to flip the statewide result, even if 100% of its votes were for the same party, this may be the least harmful option.

UNLESS - Population data could be obtained showing exactly how much population  the other counties lost when Broomfield was created. In that case, 2000 estimates of population could be interpolated proportionally among the counties, even if this would not perfectly reflect actual population values. BUT THEN would the model then infer non-existent political implications of population drop? THEN AGAIN unless Colorado were cut entirely, those population drops would be present.

ANOTHER OPTION - Broomfield County is coterminous with the city of Broomfield, which existed before the county and does have 2000 population data. BUT - Does data exist for the 2000 population of the other counties on present boundaries? Could those be interpolated?

Until I figure out the best solution, I'll cut it.


Shannon County and Oglala Lakota County, South Dakota are the same; the former changed its name to the latter in 2015. Entries with the old name will be updated to the new for consistency. The county's FIPS Code also changed at this time; we will use the new one.

In [44]:
election_pivot.loc[
    (election_pivot['County'] == 'Shannon County') & 
    (election_pivot['State'] == 'South Dakota'),
    ['County', 'FIPS Code']
] = ['Oglala Lakota County', '46102']

In [45]:
election_pivot = election_pivot[~election_pivot['County'].str.contains("Broomfield County", na=False)]

In [46]:
# Check again for missing county/year pairs. There should be none now.
expected_years = {2000, 2004, 2008, 2012, 2016, 2020}

# Group by State and County
grouped = election_pivot.groupby(['State', 'County'])

# Identify counties with missing years
missing_data = []

for (state, county), group in grouped:
    present_years = set(group['Year'])
    missing_years = expected_years - present_years
    if missing_years:
        missing_data.append({
            'State': state,
            'County': county,
            'Missing Years': sorted(missing_years)
        })

# Convert to DataFrame
missing_df = pd.DataFrame(missing_data)

missing_df

""


Success!

# FIPS Code completeness and formatting

With six elections encompassed, there should be the same number of rows for each election, and the same number of FIPS Codes across the board.

In [47]:
election_pivot.shape

(18666, 13)

In [48]:
election_pivot['Year'].value_counts()

Year
2000    3111
2004    3111
2008    3111
2012    3111
2016    3111
2020    3111
Name: count, dtype: int64

In [49]:
election_pivot['FIPS Code'].value_counts()

FIPS Code
1001.0     6
39157.0    6
39113.0    6
39115.0    6
39117.0    6
          ..
21161.0    6
21145.0    6
21147.0    6
46102      4
46102.0    2
Name: count, Length: 3112, dtype: int64

Just an integer/decimal problem. This can be fixed easily by changed the datatype of the FIPS to integer. However, we will then have to change it to a string to prevent leading zeros from being dropped in certain codes.

In [50]:
election_pivot['FIPS Code'] = election_pivot['FIPS Code'].astype(int)
election_pivot['FIPS Code'] = election_pivot['FIPS Code'].astype(str).str.zfill(5)

/var/folders/cf/4zrkk0lx1nbdx9tpklpk4jc00000gn/T/ipykernel_6169/2668210033.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  election_pivot['FIPS Code'] = election_pivot['FIPS Code'].astype(int)
/var/folders/cf/4zrkk0lx1nbdx9tpklpk4jc00000gn/T/ipykernel_6169/2668210033.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  election_pivot['FIPS Code'] = election_pivot['FIPS Code'].astype(str).str.zfill(5)


In [51]:
election_pivot

,Year,State,County,FIPS Code,Democrat Votes,Democrat Percentage,Republican Votes,Republican Percentage,Other Votes,Other Percentage,Total Votes,D-R margin,Two party vote share
0,2000,Alabama,Autauga County,01001,4942,28.72,11993,69.69,273,1.59,17208,-40.97,98.41
1,2000,Alabama,Baldwin County,01003,13997,24.78,40872,72.37,1611,2.85,56480,-47.59,97.15
2,2000,Alabama,Barbour County,01005,5188,49.91,5096,49.02,111,1.07,10395,0.89,98.93
3,2000,Alabama,Bibb County,01007,2710,38.16,4273,60.17,118,1.66,7101,-22.01,98.33
4,2000,Alabama,Blount County,01009,4977,27.69,12667,70.48,329,1.83,17973,-42.79,98.17
...,...,...,...,...,...,...,...,...,...,...,...,...,...
18667,2020,Wyoming,Sweetwater County,56037,3823,22.89,12229,73.24,646,3.87,16698,-50.35,96.13
18668,2020,Wyoming,Teton County,56039,9848,66.60,4341,29.36,598,4.04,14787,37.24,95.96
18669,2020,Wyoming,Uinta County,56041,1591,16.82,7496,79.25,372,3.93,9459,-62.43,96.07
18670,2020,Wyoming,Washakie County,56043,651,16.15,3245,80.48,136,3.37,4032,-64.33,96.63


# Our election dataset is fully cleaned!

Now we can save it to a CSV.

In [52]:
file_path = '/Users/matthewcavanaugh/Desktop/GitHub/us-presidential-elections-by-county/sources/Election CLEANED.csv'
# Save the DataFrame to CSV at the specified path, without including the index
election_pivot.to_csv(file_path, index=False)

In [53]:
loadtest = pd.read_csv("/Users/matthewcavanaugh/Desktop/GitHub/us-presidential-elections-by-county/sources/Election CLEANED.csv", dtype={'FIPS Code': str})
loadtest

,Year,State,County,FIPS Code,Democrat Votes,Democrat Percentage,Republican Votes,Republican Percentage,Other Votes,Other Percentage,Total Votes,D-R margin,Two party vote share
0,2000,Alabama,Autauga County,01001,4942,28.72,11993,69.69,273,1.59,17208,-40.97,98.41
1,2000,Alabama,Baldwin County,01003,13997,24.78,40872,72.37,1611,2.85,56480,-47.59,97.15
2,2000,Alabama,Barbour County,01005,5188,49.91,5096,49.02,111,1.07,10395,0.89,98.93
3,2000,Alabama,Bibb County,01007,2710,38.16,4273,60.17,118,1.66,7101,-22.01,98.33
4,2000,Alabama,Blount County,01009,4977,27.69,12667,70.48,329,1.83,17973,-42.79,98.17
...,...,...,...,...,...,...,...,...,...,...,...,...,...
18661,2020,Wyoming,Sweetwater County,56037,3823,22.89,12229,73.24,646,3.87,16698,-50.35,96.13
18662,2020,Wyoming,Teton County,56039,9848,66.60,4341,29.36,598,4.04,14787,37.24,95.96
18663,2020,Wyoming,Uinta County,56041,1591,16.82,7496,79.25,372,3.93,9459,-62.43,96.07
18664,2020,Wyoming,Washakie County,56043,651,16.15,3245,80.48,136,3.37,4032,-64.33,96.63


In [54]:
jack_mo = election_pivot[election_pivot['County'].str.contains('Jackson')]
jack_mo = jack_mo[jack_mo['State'].str.contains('Missouri')]
jack_mo

,Year,State,County,FIPS Code,Democrat Votes,Democrat Percentage,Republican Votes,Republican Percentage,Other Votes,Other Percentage,Total Votes,D-R margin,Two party vote share
1499,2000,Missouri,Jackson County,29095,160419,58.960000,104418,38.380000,7225,2.660000,272062,20.580000,97.340000
4612,2004,Missouri,Jackson County,29095,183654,58.119642,130500,41.298383,1839,0.581975,315993,16.821259,99.418025
7724,2008,Missouri,Jackson County,29095,210824,62.141211,124687,36.751988,3755,1.106801,339266,25.389223,98.893199
10836,2012,Missouri,Jackson County,29095,183953,59.041423,122708,39.384272,4905,1.574305,311566,19.657151,98.425695
13948,2016,Missouri,Jackson County,29095,168972,55.973976,116211,38.496270,16693,5.529754,301876,17.477706,94.470246
17060,2020,Missouri,Jackson County,29095,199842,60.001261,126535,37.991311,6686,2.007428,333063,22.009950,97.992572
